# Expected Threat (xT) on Impect SPADL

Run notebook 1 first (`1-load-and-convert-impect-data.ipynb`) so `spadl-impect.h5` exists.

xT is **provider-agnostic**: any SPADL HDF5 works. This notebook only differs in where the file lives (Impect config / open-data example).

In [ ]:
%load_ext autoreload
%autoreload 2
import socceraction.spadl as spadl
import socceraction.xthreat as xthreat

## Load SPADL actions

In [ ]:
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import tqdm
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

PROJECT_ROOT = Path('..').resolve()
CONFIG_CANDIDATES = [
    PROJECT_ROOT / 'private/impect-pipeline/config/my_iteration.json',
    PROJECT_ROOT / 'docs/documentation/data/impect_open_data.example.json',
]
CONFIG_PATH = next((p for p in CONFIG_CANDIDATES if p.is_file()), CONFIG_CANDIDATES[-1])
with open(CONFIG_PATH) as f:
    CFG = json.load(f)
OUT = PROJECT_ROOT / 'data/impect' / CFG['output_basename']
spadl_h5 = OUT / 'spadl-impect.h5'
xT_json = OUT / 'xt_model.json'
print('config:', CONFIG_PATH)
print('spadl_h5:', spadl_h5)

def games_with_actions(path):
    with pd.HDFStore(path) as store:
        games = store['games']
        ids = {int(k.rsplit('_', 1)[-1]) for k in store.keys() if k.startswith('/actions/game_')}
    return games[games.game_id.isin(ids)].sort_values('game_date').reset_index(drop=True)


In [ ]:
games = games_with_actions(spadl_h5)
if not spadl_h5.is_file():
    raise FileNotFoundError(f'Build SPADL first: {spadl_h5}')

A = []
with pd.HDFStore(spadl_h5) as store:
    for game in tqdm.tqdm(list(games.itertuples()), desc='load actions'):
        actions = store[f'actions/game_{game.game_id}']
        actions = spadl.add_names(actions)
        actions = spadl.play_left_to_right(actions, game.home_team_id)
        A.append(actions)
A = pd.concat(A, ignore_index=True)
print(len(games), 'games,', len(A), 'actions')


## Option A — Pre-trained grid (Karun Singh)

Uses the public 12×8 grid from [Karun's blog](https://karun.in/blog/expected-threat.html).

In [ ]:
# Uncomment if you hit SSL errors downloading the grid:
# import ssl
# ssl._create_default_https_context = ssl._create_unverified_context

url_grid = 'https://karun.in/blog/data/open_xt_12x8_v1.json'
xT_pretrained = xthreat.load_model(url_grid)
mov_pretrained = xthreat.get_successful_move_actions(A)
mov_pretrained['xT_value'] = xT_pretrained.rate(mov_pretrained)
mov_pretrained[['type_name', 'start_x', 'start_y', 'end_x', 'end_y', 'xT_value']].head(10)


## Option B — Train on your iteration

In [ ]:
xTModel = xthreat.ExpectedThreat(l=16, w=12)
xTModel.fit(A)
xTModel.save_model(str(xT_json))
print('saved', xT_json)


## Rate ball-progressing actions

In [ ]:
# Use the model fit on your data (switch to xT_pretrained for the public grid)
mov_actions = xthreat.get_successful_move_actions(A)
mov_actions['xT_value'] = xTModel.rate(mov_actions)
mov_actions[['type_name', 'start_x', 'start_y', 'end_x', 'end_y', 'xT_value']].head(10)


## Top players by xT (sum over successful moves)

In [ ]:
with pd.HDFStore(spadl_h5) as store:
    players = store['players'] if 'players' in store else pd.DataFrame()

top = (
    mov_actions.groupby('player_id')['xT_value']
    .agg(xT_value='sum', actions='count')
    .reset_index()
    .sort_values('xT_value', ascending=False)
)
if len(players):
    top = top.merge(players, on='player_id', how='left')
top.head(15)


## Visualize the grid (optional)

Requires `pip install matplotsoccer`.

In [ ]:
try:
    import matplotsoccer as mps
except ImportError as exc:
    raise ImportError('pip install matplotsoccer') from exc

mps.heatmap(xTModel.xT, cmap='hot', linecolor='white', cbar=True)
interp = xTModel.interpolator()
x = np.linspace(0, 105, 1050)
y = np.linspace(0, 68, 680)
mps.heatmap(interp(x, y), cmap='hot', linecolor='white', cbar=True)


## Scatter: actions colored by xT

In [ ]:
import matplotlib.pyplot as plt
try:
    import matplotsoccer as mps
except ImportError as exc:
    raise ImportError('pip install matplotsoccer') from exc

a = mov_actions.sort_values('xT_value', ascending=False)
mps.field(show=False)
plt.title('Impect actions colored by xT')
plt.scatter(a.start_x, a.start_y, c=a.xT_value, cmap='bwr_r', s=8, alpha=0.6)
plt.colorbar()
plt.show()
